# Braintrust trace review with ReactCode and Label Studio

This notebook is the companion for the **Braintrust → Label Studio Enterprise** ReactCode integration tutorial.

You will:
- generate or pull traces from Braintrust
- normalize them into a consistent trace-review schema
- create a Label Studio project via API
- import traces as tasks
- review + annotate in a custom ReactCode UI

> **Requirement:** ReactCode is available in **Label Studio Enterprise** only.

## 1) Configuration

Create a `.env` file in the repository root (or the same directory as this notebook) with the following variables:

```bash
# Label Studio Enterprise
LABEL_STUDIO_HOST=http://localhost:8080       # or your LS Enterprise instance URL
LABEL_STUDIO_API_KEY=your_label_studio_api_key

# Braintrust
BRAINTRUST_API_KEY=your_braintrust_api_key
BRAINTRUST_PROJECT_NAME=your_project_name     # project name for tracing and fetching

# OpenAI (only needed for Section 3a sample trace generation)
OPENAI_API_KEY=your_openai_api_key
```

In [46]:
import os
from dotenv import load_dotenv

# Load .env from current directory or repository root
# override=True ensures kernel picks up .env changes without restart
load_dotenv(override=True)
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), '.env'), override=True)  # fallback: repo root

# Label Studio Enterprise
LABEL_STUDIO_HOST = os.getenv('LABEL_STUDIO_HOST', 'http://localhost:8001')
LABEL_STUDIO_API_KEY = os.getenv('LABEL_STUDIO_API_KEY', '')

# Braintrust
BRAINTRUST_API_KEY = os.getenv('BRAINTRUST_API_KEY', '')
BRAINTRUST_PROJECT_NAME = os.getenv('BRAINTRUST_PROJECT_NAME', '')

# OpenAI (only needed for sample trace generation in Section 3a)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')

print('LABEL_STUDIO_HOST:', LABEL_STUDIO_HOST)
print('BRAINTRUST_PROJECT_NAME:', BRAINTRUST_PROJECT_NAME or '(not set)')
print('Has LABEL_STUDIO_API_KEY?', bool(LABEL_STUDIO_API_KEY))
print('Has BRAINTRUST_API_KEY?', bool(BRAINTRUST_API_KEY))
print('Has OPENAI_API_KEY?', bool(OPENAI_API_KEY))

LABEL_STUDIO_HOST: http://localhost:8001
BRAINTRUST_PROJECT_NAME: ls-project
Has LABEL_STUDIO_API_KEY? True
Has BRAINTRUST_API_KEY? True
Has OPENAI_API_KEY? True


## 2) Install dependencies

In [41]:
!pip -q install requests label-studio-sdk python-dotenv braintrust braintrust-api braintrust-langchain langchain langchain-openai langgraph


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 3) Label Studio ReactCode config
3-panel UI (turn list → detail → annotation) for trace review. Loaded from `label_config.py` in this directory.

In [17]:
# Load 3-panel ReactCode config (label_config.py + template.js in same directory)
_config_paths = [
  'label_config.py',
  os.path.join(os.getcwd(), 'tutorials', 'braintrust_trace_review_with_reactcode_and_label_studio', 'label_config.py'),
]
_config_path = next((p for p in _config_paths if os.path.exists(p)), None)
if _config_path:
  import importlib.util
  _spec = importlib.util.spec_from_file_location('label_config', _config_path)
  _mod = importlib.util.module_from_spec(_spec)
  _spec.loader.exec_module(_mod)
  LABEL_CONFIG_XML = _mod.LABEL_CONFIG_XML
else:
  raise FileNotFoundError('label_config.py not found. Ensure it is in the same directory as this notebook.')
print(LABEL_CONFIG_XML[:300] + '\n...')

<View>
  <ReactCode style="height: 95vh" name="trace" toName="trace" outputs='{"trace_id":"string","turn_id":"string","turn_role":"string","verdict":"string","failure_modes":"array","severity":"string","expected_behavior":"string","comments":"string"}'>
    <![CDATA[
    function TraceAnnotator({ Re
...


## 3a) Generate sample traces (optional)

If you already have traces in Braintrust, **skip this cell** — set `GENERATE_TRACES = False` and go directly to Section 4.

Otherwise, this cell creates a ReAct agent with multiple tools and runs 4 multi-turn conversations to produce realistic traces in your Braintrust project. Requires `OPENAI_API_KEY`.

In [21]:
GENERATE_TRACES = True  # Set to False if you already have traces in Braintrust

if GENERATE_TRACES:
    import braintrust
    from braintrust_langchain import BraintrustCallbackHandler, set_global_handler
    from langchain_core.tools import tool
    from langchain_core.messages import HumanMessage
    from langchain_openai import ChatOpenAI
    from langchain.agents import create_agent as create_react_agent

    if not OPENAI_API_KEY:
        raise RuntimeError('OPENAI_API_KEY is required for trace generation. Set it in your .env or set GENERATE_TRACES=False.')
    if not BRAINTRUST_PROJECT_NAME:
        raise RuntimeError('BRAINTRUST_PROJECT_NAME is required. Set it in your .env file.')

    # Initialize Braintrust logger + LangChain callback handler
    braintrust.init_logger(project=BRAINTRUST_PROJECT_NAME, api_key=BRAINTRUST_API_KEY)
    handler = BraintrustCallbackHandler()
    set_global_handler(handler)

    # --- Tools ---
    @tool
    def calculator(expression: str) -> str:
        """Evaluate a math expression. Input should be a valid Python math expression."""
        try:
            return str(eval(expression))
        except Exception as e:
            return f"Error: {e}"

    @tool
    def search_knowledge_base(query: str) -> str:
        """Search an internal knowledge base for information about company policies, products, or procedures."""
        kb = {
            "refund": "Refund policy: Full refund within 30 days of purchase. After 30 days, store credit only. Damaged items: full refund at any time with photo evidence.",
            "shipping": "Shipping: Standard (5-7 business days, free over $50), Express (2-3 days, $12.99), Overnight ($24.99). International shipping available to 40+ countries.",
            "warranty": "Warranty: 1-year limited warranty on all electronics. 2-year extended warranty available for $29.99. Covers manufacturing defects only.",
            "pricing": "Enterprise pricing: Base plan $99/mo (up to 10 users), Pro plan $249/mo (up to 50 users), Enterprise plan custom pricing. Annual billing saves 20%.",
        }
        query_lower = query.lower()
        results = [v for k, v in kb.items() if k in query_lower]
        return results[0] if results else f"No results found for: {query}"

    @tool
    def get_weather(city: str) -> str:
        """Get the current weather for a city."""
        weather_data = {
            "new york": "New York: 72°F, Partly Cloudy, Humidity 65%, Wind 8 mph SW",
            "london": "London: 58°F, Overcast, Humidity 80%, Wind 12 mph W",
            "tokyo": "Tokyo: 82°F, Clear, Humidity 55%, Wind 5 mph NE",
            "paris": "Paris: 63°F, Light Rain, Humidity 75%, Wind 10 mph NW",
        }
        return weather_data.get(city.lower(), f"Weather data not available for {city}")

    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    agent = create_react_agent(llm, [calculator, search_knowledge_base, get_weather])

    # --- 4 multi-turn conversations ---
    # Each conversation is wrapped in a Braintrust traced span to group turns.
    # The global BraintrustCallbackHandler auto-instruments all LangChain calls.
    conversations = [
        # 1: Customer support with knowledge base lookups
        [
            "What's the refund policy for a product I bought 3 weeks ago?",
            "What if the item arrived damaged? Does that change anything?",
        ],
        # 2: Multi-step calculation with follow-up
        [
            "I need to calculate the total cost for our team: 35 users on the Pro plan with annual billing. What's the yearly cost?",
            "How much would we save compared to monthly billing?",
        ],
        # 3: Weather + planning across cities
        [
            "I'm planning a trip. What's the weather like in Tokyo and Paris right now?",
            "Based on the weather, which city would be better for outdoor sightseeing today?",
        ],
        # 4: Mixed tool usage — knowledge base + calculator
        [
            "What shipping options do you have, and how much would express shipping cost for 5 orders?",
            "If I spend $250 total on products, which shipping is free? And what's the total with express for the remaining orders?",
        ],
    ]

    for i, conv_messages in enumerate(conversations, 1):
        print(f"\n--- Conversation {i} ---")

        @braintrust.traced(name=f"conversation_{i}")
        def run_conversation(messages):
            chat_history = []
            final_reply = ""
            for msg_text in messages:
                print(f"  User: {msg_text[:80]}...")
                chat_history.append(HumanMessage(content=msg_text))
                result = agent.invoke({'messages': chat_history})
                chat_history = result['messages']
                final_reply = result['messages'][-1].content
                print(f"  Assistant: {final_reply[:100]}...")
            return final_reply

        run_conversation(conv_messages)

    braintrust.flush()
    print(f'\n✓ Generated {len(conversations)} traces (1 per conversation). Proceed to Section 4.')
else:
    print('Skipped trace generation. Proceed to Section 4.')


--- Conversation 1 ---
  User: What's the refund policy for a product I bought 3 weeks ago?...
  Assistant: The refund policy states that you can receive a full refund within 30 days of purchase. Since you bo...
  User: What if the item arrived damaged? Does that change anything?...
  Assistant: If the item arrived damaged, you can still receive a full refund at any time, as long as you provide...

--- Conversation 2 ---
  User: I need to calculate the total cost for our team: 35 users on the Pro plan with a...
  Assistant: The yearly cost for 35 users on the Pro plan with annual billing is $2,390.40....
  User: How much would we save compared to monthly billing?...
  Assistant: You would save approximately $597.60 compared to monthly billing....

--- Conversation 3 ---
  User: I'm planning a trip. What's the weather like in Tokyo and Paris right now?...
  Assistant: The current weather is as follows:

- **Tokyo**: 82°F, Clear, Humidity 55%, Wind 5 mph NE
- **Paris*...
  User: Based o

## 4) Braintrust API client

Fetches traces (spans) from Braintrust using the REST API with BTQL queries. Spans are grouped by `root_span_id` into traces.

In [27]:
import requests
from braintrust_api import Braintrust
from typing import Any, Dict, List, Optional


class BraintrustClient:
    """Fetches traces from Braintrust via the REST API + BTQL."""

    API_URL = 'https://api.braintrust.dev'

    def __init__(self, api_key: str, project_name: str):
        self.api_key = api_key
        self.project_name = project_name
        self.headers = {
            'Authorization': f'Bearer {api_key}',
            'Content-Type': 'application/json',
        }
        # Resolve project ID from project name
        self.project_id = self._resolve_project_id()

    def _resolve_project_id(self) -> str:
        """Look up the project ID by name."""
        client = Braintrust(api_key=self.api_key)
        for project in client.projects.list():
            if project.name == self.project_name:
                return project.id
        raise ValueError(f'Project \"{self.project_name}\" not found in Braintrust. Check BRAINTRUST_PROJECT_NAME.')

    def _btql(self, query: str) -> List[Dict[str, Any]]:
        """Execute a BTQL query and return results."""
        r = requests.post(
            f'{self.API_URL}/btql',
            headers=self.headers,
            json={'query': query, 'fmt': 'json'},
            timeout=60,
        )
        r.raise_for_status()
        payload = r.json()
        return payload.get('data', [])

    def list_traces(self, limit: int = 20) -> List[Dict[str, Any]]:
        """Fetch recent traces (top-level spans) from the project."""
        query = f"""SELECT id, span_id, root_span_id, input, output, metadata, metrics,
                            scores, created, span_attributes, error
                     FROM project_logs('{self.project_id}')
                     WHERE span_id = root_span_id
                     ORDER BY created DESC
                     LIMIT {limit}"""
        return self._btql(query)

    def get_trace_spans(self, root_span_id: str) -> List[Dict[str, Any]]:
        """Fetch all spans belonging to a trace (by root_span_id)."""
        query = f"""SELECT id, span_id, root_span_id, span_parents, input, output,
                            metadata, metrics, scores, created, span_attributes, error
                     FROM project_logs('{self.project_id}')
                     WHERE root_span_id = '{root_span_id}'
                     ORDER BY created ASC
                     LIMIT 200"""
        return self._btql(query)


if not BRAINTRUST_API_KEY:
    raise RuntimeError('Missing BRAINTRUST_API_KEY — set it in your .env file.')
if not BRAINTRUST_PROJECT_NAME:
    raise RuntimeError('Missing BRAINTRUST_PROJECT_NAME — set it in your .env file.')

bt = BraintrustClient(BRAINTRUST_API_KEY, BRAINTRUST_PROJECT_NAME)
print(f'Braintrust client ready — project ID: {bt.project_id}')

Braintrust client ready — project ID: 5f456c27-fd45-4414-aba1-42487a82ca7f


## 5) Normalize Braintrust traces → unified schema

Converts Braintrust spans into a flat list of turns (user, assistant, tool) that the ReactCode template can render. Each turn has a `role`, `content`, and optional `tool_calls`, `usage`, and `model`.

In [28]:
import json as _json


def _to_str(x):
    """Safely convert any value to a readable string."""
    if x is None:
        return ''
    if isinstance(x, str):
        return x
    if isinstance(x, (dict, list)):
        try:
            return _json.dumps(x, indent=2, default=str)
        except Exception:
            return str(x)
    return str(x)


def _extract_content(obj):
    """Extract plain-text content from various data shapes."""
    if obj is None:
        return ''
    if isinstance(obj, str):
        return obj
    if isinstance(obj, dict):
        for key in ('content', 'text', 'input', 'output', 'result'):
            if isinstance(obj.get(key), str) and obj[key].strip():
                return obj[key]
        return _to_str(obj)
    if isinstance(obj, list):
        parts = [_extract_content(item) for item in obj if _extract_content(item).strip()]
        return '\n'.join(parts) if parts else _to_str(obj)
    return str(obj)


def normalize_braintrust_trace(root_span, all_spans):
    """Convert Braintrust spans into the unified trace schema.

    Braintrust span structure (from LangGraph + BraintrustCallbackHandler):
      conversation_N (root, type=function)
        └── LangGraph (type=task)
              ├── model (type=task)
              │     └── ChatOpenAI (type=llm)  ← input: [[messages]], output: {generations: [[{message}]]}
              ├── tools (type=task)
              │     └── search_knowledge_base (type=tool) ← input: {query}, output: {content, name}
              ├── model (type=task)
              │     └── ChatOpenAI (type=llm)
              ...

    Strategy:
      1. Sort spans by timestamp
      2. type=llm → extract user messages from input + assistant response from output
      3. type=tool → extract tool execution as a tool turn
      4. Skip structural spans (type=task, type=function)
    """
    trace_id = root_span.get('span_id') or root_span.get('id', '')

    spans_sorted = sorted(all_spans, key=lambda s: s.get('created') or '')

    turns = []
    turn_counter = 0
    seen_user_messages = set()

    def add_turn(role, content, **kwargs):
        nonlocal turn_counter
        if not content or not content.strip():
            return
        turn = {
            'turn_id': f'turn_{turn_counter}',
            'role': role,
            'content': content.strip(),
            'timestamp': kwargs.get('timestamp', ''),
        }
        for k in ('model', 'usage', 'tool_calls', 'tool_name', 'tool_input'):
            if k in kwargs and kwargs[k]:
                turn[k] = kwargs[k]
        turns.append(turn)
        turn_counter += 1

    for span in spans_sorted:
        attrs = span.get('span_attributes') or {}
        stype = (attrs.get('type') or '').lower()
        ts = span.get('created') or ''
        inp = span.get('input')
        out = span.get('output')
        span_metrics = span.get('metrics') or {}

        # ── LLM span: ChatOpenAI ──
        # input: [[{type: "human", content: "..."}, ...]]  (nested list)
        # output: {generations: [[{message: {content, tool_calls, ...}}]]}
        if stype == 'llm':
            # Extract user messages from input (unwrap nested list)
            messages = inp
            if isinstance(messages, list) and messages and isinstance(messages[0], list):
                messages = messages[0]  # unwrap outer list

            if isinstance(messages, list):
                for msg in messages:
                    if isinstance(msg, dict):
                        role = msg.get('role') or msg.get('type', '')
                        if role in ('user', 'human'):
                            content = msg.get('content', '')
                            if isinstance(content, list):
                                content = ' '.join(
                                    p.get('text', '') if isinstance(p, dict) else str(p) for p in content
                                )
                            if content and content.strip():
                                msg_key = content[:200]
                                if msg_key not in seen_user_messages:
                                    seen_user_messages.add(msg_key)
                                    add_turn('user', content, timestamp=ts)

            # Extract assistant response from output
            # Format: {generations: [[{message: {content, additional_kwargs: {tool_calls}}}]]}
            assistant_content = ''
            tool_calls = []
            if isinstance(out, dict):
                gens = out.get('generations')
                if isinstance(gens, list) and gens and isinstance(gens[0], list) and gens[0]:
                    gen = gens[0][0]
                    if isinstance(gen, dict):
                        message = gen.get('message') or gen
                        assistant_content = message.get('content', '') or ''

                        # Tool calls from additional_kwargs or direct
                        ak = message.get('additional_kwargs') or {}
                        tc_raw = ak.get('tool_calls') or message.get('tool_calls') or []
                        for tc in tc_raw:
                            if isinstance(tc, dict):
                                func = tc.get('function') or {}
                                tool_calls.append({
                                    'tool_name': func.get('name') or tc.get('name', 'unknown'),
                                    'input': _to_str(func.get('arguments') or tc.get('args', '')),
                                    'call_id': tc.get('id', ''),
                                })

            model_name = attrs.get('name') or ''
            usage = None
            if span_metrics.get('prompt_tokens') or span_metrics.get('completion_tokens'):
                usage = {
                    'input_tokens': span_metrics.get('prompt_tokens', 0),
                    'output_tokens': span_metrics.get('completion_tokens', 0),
                }

            if assistant_content and assistant_content.strip():
                add_turn(
                    'assistant', assistant_content,
                    timestamp=ts, model=model_name,
                    usage=usage,
                    tool_calls=tool_calls if tool_calls else None,
                )
            elif tool_calls:
                tool_names = ', '.join(tc['tool_name'] for tc in tool_calls)
                add_turn(
                    'assistant', f'[Calling: {tool_names}]',
                    timestamp=ts, model=model_name,
                    usage=usage,
                    tool_calls=tool_calls,
                )

        # ── Tool span ──
        # input: {query: "..."} or similar tool args
        # output: {content: "...", name: "tool_name", tool_call_id: "..."}
        elif stype == 'tool':
            tool_name = attrs.get('name') or 'unknown'
            tool_input = _to_str(inp) if inp else ''

            tool_output = ''
            if isinstance(out, dict):
                tool_output = out.get('content', '') or _extract_content(out)
            elif out:
                tool_output = _extract_content(out)

            if tool_output:
                add_turn(
                    'tool', tool_output,
                    timestamp=ts,
                    tool_name=tool_name,
                    tool_input=tool_input,
                )

    # Fallback: if no turns extracted, use root span input/output
    if not turns:
        root_input = _extract_content(root_span.get('input'))
        root_output = _extract_content(root_span.get('output'))
        if root_input:
            add_turn('user', root_input, timestamp=root_span.get('created', ''))
        if root_output:
            add_turn('assistant', root_output, timestamp=root_span.get('created', ''))

    return {
        'trace_id': str(trace_id),
        'session_id': str(trace_id),
        'metadata': {
            'name': (root_span.get('span_attributes') or {}).get('name') or root_span.get('id', ''),
            'source': 'braintrust',
            'tags': root_span.get('tags') or [],
            'start_time': root_span.get('created') or '',
            'scores': root_span.get('scores') or {},
        },
        'turns': turns,
    }


print('✓ Normalization functions defined')

✓ Normalization functions defined


## 6) Fetch, normalize, and import into Label Studio

Fetches traces from Braintrust, normalizes them, creates a Label Studio project with the ReactCode config, and imports the tasks.

In [ ]:
from label_studio_sdk import LabelStudio
from label_studio_sdk.core.request_options import RequestOptions
from typing import Any, Dict, List

# Extended timeout for large ReactCode label configs
_REQUEST_OPTS = RequestOptions(timeout_in_seconds=120)


def create_project(ls_host: str, api_key: str, title: str, label_config: str) -> int:
    client = LabelStudio(base_url=ls_host, api_key=api_key)
    project = client.projects.create(title=title, label_config=label_config, request_options=_REQUEST_OPTS)
    return int(project.id)


def import_tasks(ls_host: str, api_key: str, project_id: int, tasks: List[Dict[str, Any]]) -> Any:
    client = LabelStudio(base_url=ls_host, api_key=api_key)
    return client.projects.import_tasks(id=project_id, request=tasks, return_task_ids=True)


if not LABEL_STUDIO_API_KEY:
    raise RuntimeError('Missing LABEL_STUDIO_API_KEY — set it in your .env file.')

# 1) Fetch traces from Braintrust (most recent, limited to 20)
root_spans = bt.list_traces(limit=20)

if not root_spans:
    raise RuntimeError(
        'No traces returned from Braintrust. Ensure you have traces in your project '
        'and that BRAINTRUST_* credentials are correct. Run Section 3a to generate sample traces.'
    )

print(f'Fetched {len(root_spans)} root spans from Braintrust')

# 2) Normalize each trace — only include traces with child spans
#    (traces without child spans were logged without the BraintrustCallbackHandler
#    and won't have LLM/tool detail)
tasks: List[Dict[str, Any]] = []
skipped = 0
for root in root_spans:
    root_span_id = root.get('span_id') or root.get('root_span_id')
    if not root_span_id:
        continue
    all_spans = bt.get_trace_spans(root_span_id)

    # Skip traces that only have the root span (no child instrumentation)
    if len(all_spans) <= 1:
        skipped += 1
        continue

    normalized = normalize_braintrust_trace(root, all_spans)

    if normalized['turns']:
        tasks.append({'data': normalized})
        print(f"  ✓ Trace {root_span_id[:12]}... → {len(normalized['turns'])} turns "
              f"({sum(1 for t in normalized['turns'] if t['role']=='user')} user, "
              f"{sum(1 for t in normalized['turns'] if t['role']=='assistant')} assistant, "
              f"{sum(1 for t in normalized['turns'] if t['role']=='tool')} tool)")

if skipped:
    print(f'  (skipped {skipped} traces without child spans)')

print(f'\nPrepared {len(tasks)} tasks for import')

# 3) Create project and import
project_id = create_project(
    ls_host=LABEL_STUDIO_HOST,
    api_key=LABEL_STUDIO_API_KEY,
    title='Braintrust Trace Review (ReactCode)',
    label_config=LABEL_CONFIG_XML,
)
print(f'Created project: {project_id}')

resp = import_tasks(LABEL_STUDIO_HOST, LABEL_STUDIO_API_KEY, project_id, tasks)
print(f'Imported {len(tasks)} tasks')

print(f'\n✓ Open your project: {LABEL_STUDIO_HOST.rstrip("/")}/projects/{project_id}')

## What's next

- **Start annotating**: Open the project link above and click through traces in the ReactCode UI
- **Share with SMEs**: Invite domain experts to your Label Studio project for collaborative evaluation
- **Incremental sync**: Re-run sections 4–6 periodically to pull new traces
- **Custom taxonomy**: Edit `template.js` to add failure modes specific to your domain
- **Langfuse / LangSmith**: See companion tutorials for other observability platforms